# 1.1 原始数据理解与质量核验 · 复核

这一步当时用 `src/profile.py` 执行、`checks/verify.py` 复核，产物是 `outputs/data_profile.json`。
这个 notebook 从两份原始 CSV 重算关键数字，和当时的产物逐项对账，不改任何历史文件。


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
WINES = {"red": "红酒", "white": "白酒"}
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
STEP = ROOT / "steps/01_数据预处理/1.1_原始数据理解与质量核验"
profile = json.loads((STEP / "outputs/data_profile.json").read_text(encoding="utf-8"))
run = dsflow.start_run("1.1", project=ROOT, hypothesis="从原始 CSV 重算行数、字段、空值、同值行和 quality 分布，与 1.1 当时的产物一致")
raw, text = {}, {}
for w, name in WINES.items():
    path = ROOT / f"data/winequality-{w}.csv"
    run.log_input(path, name=name)
    raw[w] = pd.read_csv(path, sep=";")
    text[w] = pd.read_csv(path, sep=";", dtype=str)
    print(f"{name}：{len(raw[w]):,} 行 × {raw[w].shape[1]} 列；表头 = 字段说明里的 11 个理化指标 + quality：{list(raw[w].columns) == FEATURES + ['quality']}")


红酒：1,599 行 × 12 列；表头 = 字段说明里的 11 个理化指标 + quality：True
白酒：4,898 行 × 12 列；表头 = 字段说明里的 11 个理化指标 + quality：True


In [2]:
rows = []
for w, name in WINES.items():
    df, tx, p = raw[w], text[w], profile["files"][w]
    same_rows = int(tx.duplicated().sum())          # 12 个字段原始文本完全相同的行，每组不计首次出现
    rows.append({"酒类": name, "行数": len(df), "空值": int(df.isna().sum().sum()), "quality 超出 0～10": int((~df["quality"].between(0, 10)).sum()),
                 "同值行": same_rows, "当时记的行数": p["rows"], "当时记的同值行": p["exact_duplicate_rows_beyond_first"]})
    assert len(df) == p["rows"] and same_rows == p["exact_duplicate_rows_beyond_first"]
    assert all(df[c].isna().sum() == 0 and p["empty_by_field"][c] == 0 for c in df.columns)
check = pd.DataFrame(rows)
print(check.to_string(index=False))
print("和 1.1 当时的产物逐项一致。")


酒类   行数  空值  quality 超出 0～10  同值行  当时记的行数  当时记的同值行
红酒 1599   0                0  240    1599      240
白酒 4898   0                0  937    4898      937
和 1.1 当时的产物逐项一致。


In [3]:
dist = []
for w, name in WINES.items():
    counts = raw[w]["quality"].value_counts().sort_index()
    for q, n in counts.items():
        dist.append({"酒类": name, "quality": int(q), "行数": int(n), "占比": f"{n / len(raw[w]):.2%}"})
    assert int(counts.sum()) == len(raw[w])
dist = pd.DataFrame(dist)
print(dist.to_string(index=False))
run.log_metrics({"红酒行数": len(raw["red"]), "白酒行数": len(raw["white"]), "红酒同值行": int(text["red"].duplicated().sum()), "白酒同值行": int(text["white"].duplicated().sum())})
run.set_conclusion("重算的行数、空值、同值行、quality 分布与 1.1 当时的 data_profile.json 逐项一致", validity="有效")
run.end()


酒类  quality   行数     占比
红酒        3   10  0.63%
红酒        4   53  3.31%
红酒        5  681 42.59%
红酒        6  638 39.90%
红酒        7  199 12.45%
红酒        8   18  1.13%
白酒        3   20  0.41%
白酒        4  163  3.33%
白酒        5 1457 29.75%
白酒        6 2198 44.88%
白酒        7  880 17.97%
白酒        8  175  3.57%
白酒        9    5  0.10%
